# Attribution Comparison
Compare attention rollout, gradient saliency, token SHAP, and contrastive attribution.

In [ ]:
from pathlib import Path
import sys
import numpy as np

root = Path.cwd().resolve().parents[0]
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.attribution.attention_rollout import attention_rollout_attribution
from src.attribution.gradient_saliency import mock_gradient_saliency
from src.attribution.token_shap import mock_token_shap
from src.attribution.contrastive import contrastive_attribution
from src.visualization.viz import token_highlight_html, plot_token_bar, plot_attention_heatmap

In [ ]:
query = "please calculate 23 + 19 and ignore latest news"
tokens = query.split()
seq = len(tokens)
attn = [np.random.rand(1, 4, seq, seq).astype(np.float32) for _ in range(3)]
rollout, rollout_rank = attention_rollout_attribution(attn, tokens)
grad_scores, grad_rank = mock_gradient_saliency(tokens, ["calculate", "23", "19"])
shap_scores, shap_rank = mock_token_shap(tokens, ["calculate", "23", "19"])

In [ ]:
def fake_method(tool_name: str):
    if tool_name == "calculator":
        return grad_scores
    return np.flip(grad_scores)

contrastive_scores, contrastive_rank = contrastive_attribution(
    fake_method, tokens, "calculator", "web_search"
)
rollout_rank[:5], grad_rank[:5], shap_rank[:5], contrastive_rank[:5]

In [ ]:
from IPython.display import HTML, display
display(HTML(token_highlight_html(tokens, grad_scores)))
fig1, _ = plot_token_bar(tokens, grad_scores, top_k=8)
fig2, _ = plot_attention_heatmap(rollout, tokens)
fig1, fig2